# Correlation Visualization for R2R Manuscript

2026 version

In [1]:
%cd ../..

/home/bhkuser/bhklab/katy/readii_2_roqc


In [2]:
from damply import dirs
import pandas as pd
import itertools
import plotly.graph_objects as go 
from plotly.subplots import make_subplots
import numpy as np
import re


from readii_2_roqc.utils.loaders import load_signature_config, load_dataset_config, load_clinical_data
from readii_2_roqc.utils.analysis import clinical_data_setup, prediction_data_splitting

In [3]:
datasource = "TCIA"
dataset = "RADCURE_windowed"

feature_method = "pyradiomics"
feature_settings = "linear_all_images_features"

image_type_a = "original_full"
image_type_b = "randomized_roi"

In [4]:
dataset_config, dataset_name, full_data_name = load_dataset_config(dataset)
split = dataset_config['ANALYSIS']['TRAIN_TEST_SPLIT']['split']

2026-05-01 | WARNING | Environment variable 'CONFIG' is not set. Using default path relative to project root.


In [8]:
if split:
    clinical_data = clinical_data_setup(dataset_config, 
                                        full_data_name,
                                        split)
    clinical_data = clinical_data[clinical_data.index.notnull()]
    test_samples = pd.Series(clinical_data.index)

In [11]:
# Save out test samples if needed for other things
# test_samples.to_csv(dirs.PROCDATA / f"{datasource}_{dataset}/test_samples.csv", index=False)

## Big correlation plots

In [ ]:
self_corr_path = dirs.RESULTS / f"{datasource}_{dataset}" / "correlation" / "self" / feature_method / feature_settings / f"{image_type_a}_pearson.csv"
cross_corr_path = dirs.RESULTS / f"{datasource}_{dataset}" / "correlation" / "cross" / feature_method / feature_settings / f"{image_type_b}_vs_{image_type_a}_pearson.csv"

corr_df = pd.read_csv(self_corr_path, index_col=0)
# cross_corr_df = pd.read_csv(cross_corr_path, index_col=0)

### Original Features first

In [ ]:
feature_types = corr_df.columns.copy().to_series(name="feature_types")
noshape_feature_types = feature_types.drop(feature_types.filter(regex="original_shape_*", axis=0).to_list())

In [ ]:
noshape_feature_types.str.split("_", n=2)

In [ ]:
feature_classes = [
    "firstorder",
    "glcm",
    "glrlm",
    "glszm",
    "gldm",
    "ngtdm"
]

In [ ]:
class_avg_vol_corr = {}

for feat_class in feature_classes:
    # Get just the specified feature_class features' correlations with the shape features
    subset_features = corr_df.filter(regex="original_shape_*", axis=0).filter(regex=f"{feat_class}_*", axis=1)
    class_avg_vol_corr[feat_class] = subset_features.loc['original_shape_MeshVolume'].abs().mean()

In [ ]:
class_avg_vol_corr

# Features as a function of volume

Borrowed from Velichko et al.

1. Load in the features from each image type 
2. Plot selected features vs. volume as scatter 
3. Colour each image type differently

In [6]:
signature_name = 'aerts_original'

In [7]:
REGIONS = {'full', 'roi', 'non_roi'}
PERMUTATIONS = {'randomized', 'sampled', 'shuffled'}

image_types = itertools.chain([('original', 'full')],itertools.product(PERMUTATIONS,REGIONS))

feature_path = dirs.RESULTS / f"{datasource}_{dataset}" / "features" / feature_method / "original_512_512_n" / feature_settings 

feature_sets = {}

for itype in image_types:
    itype_str = '_'.join(itype)
    itype_features = pd.read_csv(feature_path / f"{itype_str}_features.csv", index_col=0)
    
    if split:
        feature_sets[itype_str] = itype_features.loc[test_samples]
    else:
        feature_sets[itype_str] = itype_features

2026-04-30 | WARNING | Environment variable 'RESULTS' is not set. Using default path relative to project root.


## Select signature features + MeshVolume

In [8]:
mesh_vol = feature_sets['original_full']['original_shape_MeshVolume']
log_mesh_vol = np.log(mesh_vol)

In [9]:
# Load config for signature
signature_name = "aerts_original"
signature = load_signature_config(signature_name)

sig_feats = signature.index.to_list()

# For each feature in the signature, create a dataframe where each column is an image type and the rows are the select feature values for all samples
sig_feature_sets = {}
for feat in sig_feats:
    all_itypes_feat_dict = {itype: feature_set.loc[:,feat] for itype,feature_set in feature_sets.items()}
    select_feat_df = pd.DataFrame.from_dict(all_itypes_feat_dict).sort_index(axis=1)
    # replace non_roi with background
    select_feat_df = select_feat_df.rename(columns=lambda x: re.sub('non_roi', 'background',x))
    sig_feature_sets[feat] = select_feat_df

In [16]:
colours = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']
region = 'full'
image_types=[f'randomized_{region}', f'sampled_{region}', f'shuffled_{region}', 'original_full']

plot_vol = mesh_vol

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=(sig_feats),
)

show_legend = True
for idx, feat_name in enumerate(sig_feature_sets.keys()):
    plot_feat = sig_feature_sets[feat_name]
    
    # Plot permutation features
    for col_idx, itype in enumerate(image_types):
        fig.add_trace(
            go.Scatter(
                x=plot_vol,
                y=plot_feat[itype],
                mode='markers',
                marker=dict(
                    size=6,
                    color=colours[col_idx]
                ),
                opacity=0.8,
                name=itype,
               legendgroup='group1',
               showlegend=show_legend
        ),
        row=1,
        col=idx+1
    )
    show_legend = False

fig.update_layout(
    title=signature_name,
    title_subtitle=dict(
        text=dataset),
    width=2500,
    height=550,
    
    
)

fig.update_xaxes(title_text="Mesh_Volume", title_font=dict(size=12))
fig.update_annotations(font_size=14)
fig.show()

In [11]:
def plot_feat_vs_volume(plot_feature_df:pd.DataFrame,
                        plot_vol:pd.Series,
                        image_types:list,
                        feature_name:str | None = None,
                        y_min:int | float | None = None,
                        y_max:int | float | None = None,
                        log_vol:bool = False,
                        log_feat:bool = False,
                        ) -> go.Figure:
    # colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52']
    fig = go.Figure()

    for itype in image_types:
        fig.add_trace(
            go.Scatter(
                x=plot_vol,
                y=plot_feature_df[itype],
                mode='markers',
                marker=dict(
                    size=6,
                ),
                name=itype,
                opacity=0.8
            )
        )

    fig.update_layout(
        title=feature_name,
        height=700,
        width=900
    )

    if log_vol:
        fig.update_xaxes(type="log")

    fig.update_yaxes(range=[y_min,y_max])
    if log_feat:
        fig.update_yaxes(type="log")

    return fig

In [20]:
# feature = signature.index[3]

region = 'full'
image_types=[f'randomized_{region}', f'sampled_{region}', f'shuffled_{region}', 'original_full']
log_vol = True
log_feat = True

fig_list = []
for idx, feature in enumerate(signature.index):

    plot_feat_df = sig_feature_sets[feature]
    log_fig = go.Figure()
    log_fig = plot_feat_vs_volume(plot_feat_df, mesh_vol, image_types, feature, log_vol = log_vol, log_feat=log_feat)
    log_fig.update_layout(
        title_subtitle=dict(text=dataset)
    )

    fig_list.append(log_fig)


In [21]:
for fig in fig_list:
    fig.show()

In [ ]:
# feature_str = f"{feature}_{region}"
# if log_feat:
#     feature_str = f"log_{feature_str}"
# if log_vol:
#     feature_str = f"logvol_{feature_str}"

# image_out_path = dirs.RESULTS / full_data_name / "visualization" / signature_name / split / "scatter_vs_volume" / f"{feature_str}.png"
# image_out_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
fig = make_subplots(
    rows=3, cols=4,
    # x_title='Mesh Volume',
    subplot_titles=(sig_feats))


show_legend = True
for idx, feat_name in enumerate(sig_feature_sets.keys()):
    plot_feat = sig_feature_sets[feat_name]

    full_feat_set = plot_feat.columns.str.contains('full')
    roi_feat_set = plot_feat.columns.str.contains('roi')
    background_feat_set = plot_feat.columns.str.contains('background')

    # Set first bool in roi and background list to True so 'original_full' is plotted
    roi_feat_set[0] = True 
    background_feat_set[0] = True

    for row_idx, feat_subset in enumerate([full_feat_set, roi_feat_set, background_feat_set]):
        # Plot the full region negative control points
        for itype in plot_feat.loc[:,feat_subset]:
            fig.add_trace(
                go.Scatter(
                    x = mesh_vol,
                    y = plot_feat[itype],
                    mode='markers',
                    marker=dict(
                        size=6
                    ),
                    name=itype,
                    legendgroup='group2',
                    showlegend=show_legend
                    ),
                row=row_idx+1,
                col=idx+1,
            )
    

    
    if show_legend:
        show_legend=False

fig.update_layout(
    title=signature_name,
    title_subtitle=dict(
        text=dataset),
    width=2500,
    height=550
    
)

fig.update_xaxes(title_text="Mesh_Volume", title_font=dict(size=12))
fig.update_annotations(font_size=14)
fig.show()

In [ ]:
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=(sig_feats))

show_legend=True

for idx, feat_name in enumerate(sig_feature_sets.keys()):
    plot_feat = sig_feature_sets[feat_name]

    for itype in ['original_full']:
        fig.add_trace(
            go.Scatter(
                x = mesh_vol,
                y = plot_feat[itype],
                mode='markers',
                name=itype,
                marker=dict(
                    color='#636EFA'
                ),
                legendgroup='group1',
                showlegend=show_legend
                ),
            row=1,
            col=idx+1
        )
        if show_legend:
            show_legend=False

fig.update_annotations(font_size=12)
fig.show()